# 🎓 Student Risk Prediction — Exploratory Data Analysis

An end-to-end EDA for the AI-Powered Student Risk Prediction System: understand the data, inspect cleaning, and visualise relationships before model training.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import utils

In [ ]:
utils.ensure_sample_data(utils.Path("data/students.csv"))
df = utils.load_data("data/students.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Dtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows (excluding Student_ID):")
cols = [c for c in df.columns if c != utils.STUDENT_ID]
print(df.duplicated(subset=cols).sum())

In [ ]:
df.describe().T

In [ ]:
df_clean, prep = utils.clean_data(df)
print(f"After cleaning: {df_clean.shape}")
print("\nRisk distribution:")
print(df_clean[utils.TARGET].value_counts())
print(df_clean[utils.TARGET].value_counts(normalize=True))

In [ ]:
risk_counts = df_clean[utils.TARGET].value_counts().rename(index={0: "Safe", 1: "At Risk"})
fig = px.bar(risk_counts, color=risk_counts.index,
             color_discrete_map={"Safe": "#2dd4bf", "At Risk": "#ff4d6d"},
             title="Risk Target Distribution")
fig.show()

In [ ]:
fig = px.histogram(df_clean, x="Attendance", color="Risk",
                   nbins=25, title="Attendance Distribution",
                   color_discrete_map={0: "#2dd4bf", 1: "#ff4d6d"},
                   labels={"Risk": "Risk Level"})
fig.show()

In [ ]:
fig = go.Figure()
for col, color in [("Assignment_Score", "#4da3ff"), ("Quiz_Score", "#22d3ee"),
                   ("Midterm", "#6c4dff"), ("Final", "#b32fff")]:
    fig.add_trace(go.Box(y=df_clean[col], name=col, marker_color=color))
fig.update_layout(title="Marks Distribution", yaxis_title="Score")
fig.show()

In [ ]:
fig = px.imshow(df_clean[utils.NUMERICAL_FEATURES + [utils.TARGET]].corr(),
                 text_auto=".2f", color_continuous_scale="RdYlBu_r",
                 aspect="auto", title="Correlation Heatmap", zmin=-1, zmax=1)
fig.show()

In [ ]:
fig = px.bar(df_clean.groupby("Risk")["GPA"].mean().rename(index={0: "Safe", 1: "At Risk"}),
             color=df_clean.groupby("Risk")["GPA"].mean().index,
             title="Average GPA by Risk Level",
             labels={"value": "Avg GPA", "index": "Risk Level"})
fig.show()

In [ ]:
fig = px.bar(df_clean.groupby("Risk")["Course_Completion"].mean().rename(index={0: "Safe", 1: "At Risk"}),
             color=df_clean.groupby("Risk")["Course_Completion"].mean().index,
             title="Average Course Completion by Risk Level",
             labels={"value": "Avg % Completion", "index": "Risk Level"})
fig.show()

In [ ]:
fig = px.bar(df_clean.groupby("Teacher_Feedback")["Risk"].mean().reset_index(),
             x="Teacher_Feedback", y="Risk", title="At-Risk Rate by Teacher Feedback",
             labels={"Risk": "At-risk rate"})
fig.show()

## Key Takeaways

- **Risk imbalance**: the target is imbalanced (~28% at risk) — use `stratify` when splitting and report precision/recall, not accuracy alone.
- **Strong correlates**: GPA, marks and attendance drive risk the most; LMS engagement features add signal.
- **Cleaning matters**: the pipeline removes duplicates, fills missing values and caps outliers with the IQR method before training.
- **Next step**: train Random Forest / Decision Tree / Logistic Regression in `train.py` and compare with ROC-AUC.